**Relaxation Regression Test**

**0. What is tested**

Two, three, and four interacting electron spins with random anisotropic g-tensors, exchange and dipolar couplings, and zero-field splitting for spin-1 centers. Each fixture produces `sample_1`, `sample_2`, `sample_mul`, and `sample_direct_sum`; each sample receives two independent contexts in the `multiplet`, `zfs`, or `product` basis.

The seven context compositions are tested for all eight combinations of secular/nonsecular, thermal correction on/off, and external decay on/off. Thermal correction is applied only when MaRs `EvolutionSuper` constructs the relaxation generator.

The named tests check dimensions, complete positivity, trace preservation or loss, Hermiticity, Boltzmann detailed balance and thermalization, and basis covariance.

See `README.md` for the reproduced failure, both library fixes and verification results.


**1. Imports and settings**

Use the MaRs implementation of superoperator construction, propagation, Choi diagnostics, and thermal conversion.

In [1]:
import os
import sys
from itertools import product

import numpy as np
import torch

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))

from mars import constants
from mars.population import check_transforms as checks
from mars.population import choi_transforms as choi
from mars.population import transform

from sample_generators import make_sample_case
from context_generators import generate_contexts
from composer import compose_cases
from utils import get_superoperators

SEED = 42
DTYPE = torch.float64
DEVICE = torch.device("cpu")
FIELD = 0.34
TEMPERATURE = 3.0
LONG_TIME = 0.06
SHORT_TIME = 1e-8
TOLERANCE = 1e-7
THERMAL_TOLERANCE = 1e-5

**2. Helper functions**

For each spin configuration, build one interacting sample fixture and two fresh contexts per sample for each secular/decay setting. The same contexts are evaluated both with and without thermal correction. Cases are yielded one at a time to limit memory use.

In [2]:
def iter_test_cases(spins: tuple[float, ...], partition: int):
    """Generate all 56 compositions and relaxation settings for one fixture."""
    rng = np.random.default_rng(SEED)
    samples = make_sample_case(
        rng, spins=spins, partition=partition,
        dtype=DTYPE, device=DEVICE, mesh=(5, 5),
    )
    for secular, decay in product((False, True), repeat=2):
        contexts = generate_contexts(
            samples, rng, secular=secular, decay=decay,
            dtype=DTYPE, device=DEVICE,
        )
        for composed in compose_cases(samples, contexts):
            basis_seed = int(rng.integers(0, 2**31 - 1))
            for corrected in (False, True):
                operators = get_superoperators(
                    composed.sample,
                    composed.context,
                    magnetic_field_t=FIELD,
                    temperature_k=TEMPERATURE,
                    thermally_corrected=corrected,
                )
                yield {
                    "composed": composed,
                    "operators": operators,
                    "secular": secular,
                    "decay": decay,
                    "corrected": corrected,
                    "basis_seed": basis_seed,
                    "description": samples.description,
                }


def case_label(case: dict) -> str:
    """Describe the sample composition and three relaxation switches."""
    return (
        f"{case['composed'].description}; "
        f"secular={case['secular']}, "
        f"thermal={case['corrected']}, decay={case['decay']}"
    )

**3. Tests**

**3.1 Composition dimensions**

The energy Hilbert space and all four superoperators must have matching dimensions.

In [3]:
def test_composition_dimensions(case: dict) -> None:
    """Check that every superoperator matches the sample dimension."""
    operators = case["operators"]
    dimension = operators["energies_hz"].numel()
    for part in ("thermal", "driven", "relaxation", "full"):
        assert operators[part].shape == (dimension**2, dimension**2), part
    initial = operators["initial_density"]
    if initial is not None:
        assert initial.shape == (dimension, dimension)

**3.2 Complete positivity**

Require generator and finite-time complete positivity for secular cases and uncorrected Lindblad cases. The driven contribution is always checked. Population-only thermal correction is allowed to break complete positivity for nonsecular cases: its thermal/combined generator and finite-time Choi margins are recorded in `case["cp_diagnostics"]` without requiring positivity. Other physical checks remain enabled.


In [4]:
def test_complete_positivity(case: dict) -> None:
    """Require CP where guaranteed; record corrected nonsecular diagnostics."""
    operators = case["operators"]
    # Population-only thermal correction need not preserve CP outside the
    # secular approximation. Uncorrected and driven Lindblad parts still must.
    require_corrected_cp = case["secular"] or not case["corrected"]
    diagnostics = case.setdefault("cp_diagnostics", {})
    for part in ("thermal", "driven", "relaxation"):
        generator = operators[part]
        valid = checks.is_completely_positive_generator(generator, atol=TOLERANCE)
        margin = choi.generator_complete_positivity_margin(generator).item()
        diagnostics[part] = margin
        if part == "driven" or require_corrected_cp:
            assert bool(valid.item()), f"{part}: Choi margin {margin:.2e}"
    propagator = transform.propagator_from_generator(operators["full"], SHORT_TIME)
    choi_matrix = choi.superoperator_to_choi(propagator)
    minimum = choi.choi_eigenvalues(choi_matrix).min().item()
    diagnostics["finite_time"] = minimum
    if require_corrected_cp:
        assert minimum >= -TOLERANCE, f"Finite-time Choi eigenvalue {minimum:.2e}"


**3.3 Trace preservation**

Without external decay, trace is conserved. With external decay, the trace-loss operator must be negative semidefinite and finite-time evolution must not increase trace.

In [5]:
def test_trace_preservation(case: dict) -> None:
    """Check trace conservation or nonincreasing trace with external decay."""
    operators = case["operators"]
    for part in ("thermal", "driven", "relaxation", "full"):
        generator = operators[part]
        if not case["decay"]:
            error = checks.generator_trace_preservation_deviation(generator).item()
            assert error <= TOLERANCE, f"{part}: trace error {error:.2e}"
        else:
            rate_operator = checks.extract_trace_rate_matrix(generator)
            maximum = torch.linalg.eigvalsh(choi.hermitian_part(rate_operator)).max().item()
            assert maximum <= TOLERANCE, f"{part}: trace grows by {maximum:.2e}"

    propagator = transform.propagator_from_generator(operators["full"], SHORT_TIME)
    if not case["decay"]:
        reduced = choi.choi_partial_trace_system(choi.superoperator_to_choi(propagator))
        identity = torch.eye(reduced.shape[-1], dtype=reduced.dtype, device=reduced.device)
        error = (reduced - identity).abs().max().item()
        assert error <= TOLERANCE, f"Finite-time trace error {error:.2e}"
    else:
        reduced = choi.choi_partial_trace_system(choi.superoperator_to_choi(propagator))
        identity = torch.eye(reduced.shape[-1], dtype=reduced.dtype, device=reduced.device)
        minimum = torch.linalg.eigvalsh(choi.hermitian_part(identity - reduced)).min().item()
        assert minimum >= -TOLERANCE, f"Finite-time trace gain {minimum:.2e}"

**3.4 Hermiticity preservation**

Thermal, driven, combined, and full generators must preserve Hermitian density matrices for all relaxation settings.

In [6]:
def test_hermiticity_preservation(case: dict) -> None:
    """Check Hermiticity preservation using native Choi diagnostics."""
    operators = case["operators"]
    for part in ("thermal", "driven", "relaxation", "full"):
        valid = checks.is_hermiticity_preserving(operators[part], atol=TOLERANCE)
        assert bool(valid.item()), part
    propagator = transform.propagator_from_generator(operators["full"], SHORT_TIME)
    assert bool(checks.is_hermiticity_preserving(propagator, atol=TOLERANCE).item())

**3.5 Boltzmann relaxation**

MaRs converts Hamiltonian energies to Kelvin. For corrected thermal rates, test detailed balance. Driving and external decay are excluded from this equilibrium comparison.

In [7]:
def test_boltzmann_relaxation(case: dict, show_plot: bool = False) -> None:
    """Check thermal detailed balance and suitable long-time Gibbs convergence."""
    if not case["corrected"]:
        return

    operators = case["operators"]
    generator = operators["thermal"]
    populations = torch.softmax(
        constants.unit_converter(-operators["energies_hz"], "Hz_to_K") / TEMPERATURE,
        dim=-1,
    )
    rates = transform.extract_transition_matrix_from_superoperator(generator)
    flux = rates * populations.unsqueeze(0)
    imbalance = (flux - flux.transpose(-2, -1)).abs()
    imbalance.fill_diagonal_(0.0)
    scale = max(1.0, flux.abs().max().item())
    assert imbalance.max().item() <= TOLERANCE * scale, "Detailed balance failed"

    name = case["composed"].description.split(" :", 1)[0]
    if (
        not case["secular"] or case["decay"]
        or not case["composed"].unique_thermal_equilibrium
        or name not in ("sample_1", "sample_2", "sample_mul")
    ):
        return

    rho_gibbs = torch.diag(populations).to(generator.dtype)
    rho_residual = transform.Liouvilleator.unvec(
        generator @ transform.Liouvilleator.vec(rho_gibbs)
    )
    assert rho_residual.abs().max().item() <= THERMAL_TOLERANCE, "Gibbs state is not stationary"

    rho_initial = operators["initial_density"]
    assert rho_initial is not None, "Initial density is missing"
    rho_initial = rho_initial.to(generator.dtype)
    rho_initial = rho_initial / rho_initial.trace().real
    propagator = transform.propagator_from_generator(generator, LONG_TIME)
    rho_final = transform.Liouvilleator.unvec(
        propagator @ transform.Liouvilleator.vec(rho_initial)
    )
    distance = torch.linalg.matrix_norm(rho_final - rho_gibbs).item()
    assert distance <= THERMAL_TOLERANCE, f"Gibbs distance {distance:.2e}"

    if show_plot:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 3.5))
        for density, style, label in (
            (rho_initial, "o--", "Initial"),
            (rho_final, "s-", "Long-time"),
            (rho_gibbs, "x:", "Boltzmann"),
        ):
            ax.plot(density.diagonal().real.cpu().numpy(), style, label=label)
        ax.set(xlabel="Energy eigenstate", ylabel="Population", title=name)
        ax.legend()
        plt.show()

**3.6 Basis covariance**

A seeded random unitary changes the representation, not physical positivity, Hermiticity, or the trace-loss spectrum.

In [8]:
def test_basis_covariance(case: dict) -> None:
    """Check native Liouville basis transformations."""
    generator = case["operators"]["relaxation"]
    dimension = case["operators"]["energies_hz"].numel()
    rng = np.random.default_rng(case["basis_seed"])
    random_matrix = rng.normal(size=(dimension, dimension)) + 1j * rng.normal(size=(dimension, dimension))
    unitary, _ = np.linalg.qr(random_matrix)
    old_basis = torch.eye(dimension, dtype=generator.dtype, device=generator.device)
    new_basis = torch.tensor(unitary, dtype=generator.dtype, device=generator.device)
    coefficients = transform.compute_liouville_basis_transformation(old_basis, new_basis)
    rotated = transform.transform_superop_to_new_basis(generator, coefficients)

    for check in (checks.is_completely_positive_generator, checks.is_hermiticity_preserving):
        original = bool(check(generator, atol=TOLERANCE).item())
        changed = bool(check(rotated, atol=TOLERANCE).item())
        assert original == changed, f"{check.__name__} depends on basis"

    trace_original = checks.extract_trace_rate_matrix(generator)
    trace_rotated = checks.extract_trace_rate_matrix(rotated)
    eigenvalues_original = torch.linalg.eigvalsh(choi.hermitian_part(trace_original))
    eigenvalues_rotated = torch.linalg.eigvalsh(choi.hermitian_part(trace_rotated))
    assert torch.allclose(eigenvalues_original, eigenvalues_rotated, atol=TOLERANCE)

**4. Test cases**

Each spin pair in the coupled sample has independently randomized exchange and anisotropic coupling. Spin-1 centers also have randomized ZFS.

| Case | Spins | Partition | Dimensions: sample_1, sample_2, coupled, direct sum |
|---|---|---|---|
| 1 | (½, 1) | 1 + 1 | 2, 3, 6, 5 |
| 2 | (1, 1) | 1 + 1 | 3, 3, 9, 6 |
| 4 | (½, ½, ½, ½) | 2 + 2 | 4, 4, 16, 8 |

In [9]:
TEST_CASES = [
    {"spins": (0.5, 1.0), "partition": 1},
    {"spins": (1.0, 1.0), "partition": 1},
    {"spins": (0.5, 0.5, 0.5, 0.5), "partition": 2},
]

**5. Run the tests**

In [10]:
def run_tests(test_cases: list[dict] = TEST_CASES, show_plots: bool = False) -> None:
    """Run all six tests on all spin and relaxation configurations."""
    total = 0
    nonsecular_cp_violations = 0
    for settings in test_cases:
        count = 0
        for case in iter_test_cases(**settings):
            if count == 0:
                print(f"Fixture: {case['description']}")
            label = case_label(case)
            try:
                test_composition_dimensions(case)
                test_complete_positivity(case)
                test_trace_preservation(case)
                test_hermiticity_preservation(case)
                plot = (
                    show_plots and case["secular"] and not case["decay"]
                    and case["corrected"] and
                    case["composed"].description.startswith("sample_1 :")
                )
                test_boltzmann_relaxation(case, show_plot=plot)
                test_basis_covariance(case)
            except AssertionError as exc:
                raise AssertionError(f"{label}: {exc}") from exc
            if not case["secular"] and case["corrected"]:
                margins = case["cp_diagnostics"]
                if any(margins[part] < -TOLERANCE for part in ("thermal", "relaxation", "finite_time")):
                    nonsecular_cp_violations += 1
                    print(f"Expected nonsecular CP violation: {label}; "
                          f"thermal margin={margins['thermal']:.2e}")
            count += 1
            print(f"Passed: {label}")
        assert count == 56, f"Expected 56 cases, received {count}"
        total += count
        print(f"Passed all {count} cases for spins={settings['spins']}")
    print(f"All {total} relaxation cases passed under the stated assumptions using seed {SEED}.")
    print(f"Corrected nonsecular cases with allowed CP violations: {nonsecular_cp_violations}")

In [ ]:
run_tests()